# Análise Visual do Mercado Global de Video Games (1980–2016)

Este notebook investiga padrões históricos, diferenças culturais de consumo e a concentração de vendas na indústria de jogos eletrônicos a partir da base pública **Video Game Sales** (compilada por Gregory Smith a partir do portal *VGChartz* e disponibilizada no [Kaggle](https://www.kaggle.com/datasets/gregorut/videogamesales)).

## Sobre o Conjunto de Dados
O arquivo `vgsales.csv` reúne jogos com vendas globais superiores a 100 mil cópias físicas e contém 11 variáveis:
- `Rank`: posição no ranking geral de vendas.
- `Name`: título do jogo.
- `Platform`: console ou plataforma de lançamento (ex.: PS2, X360, Wii, DS, PC).
- `Year`: ano de lançamento do título.
- `Genre`: gênero principal (Action, Sports, Shooter, Role-Playing, etc.).
- `Publisher`: empresa responsável pela publicação.
- `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`: vendas em milhões de cópias na América do Norte, Europa, Japão e demais regiões.
- `Global_Sales`: somatório global de vendas em milhões de unidades.

## Perguntas de Investigação
1. Como o volume de vendas e lançamentos evoluiu ao longo das gerações de consoles entre os anos 1980 e 2016?
2. O perfil de consumo por gênero no Japão segue o mesmo padrão da América do Norte e da Europa?
3. Quais plataformas e franquias sustentam o volume de vendas dentro de cada gênero?
4. As vendas são bem distribuídas entre os lançamentos ou o mercado funciona no modelo de cauda longa (poucos sucessos concentrando quase tudo)?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# carregando o dataset local
df = pd.read_csv("vgsales.csv")

print(f"Total de registros: {len(df)} | Colunas: {df.shape[1]}")
print("\nContagem de valores ausentes por coluna:")
print(df.isna().sum())

df.head(10)

### Inspeção Inicial e Diagnóstico de Valores Ausentes

A base original possui **16.598 registros** e 11 colunas. Apenas duas colunas apresentam valores nulos: `Year` (271 registros sem ano de lançamento, cerca de 1,6% da base) e `Publisher` (58 registros, cerca de 0,3%).

Para não desperdiçar dados na análise por gênero e plataforma, optamos por manter todas as colunas originais, preenchendo os 58 casos de `Publisher` ausente como `"Desconhecido"`. Já os 271 registros sem ano válido precisam ser removidos para evitar distorções nos gráficos temporais. Além disso, verificamos que a raspagem original dos dados ocorreu no final de 2016, deixando apenas 4 registros isolados entre 2017 e 2020; por isso, delimitamos o recorte histórico entre **1980 e 2016**.

In [ ]:
# mantendo todas as colunas e tratando os valores ausentes
df_limpo = df.copy()
df_limpo["Publisher"] = df_limpo["Publisher"].fillna("Desconhecido")

# filtrando registros com ano valido ate 2016
df_limpo = df_limpo.dropna(subset=["Year"]).copy()
df_limpo["Year"] = df_limpo["Year"].astype(int)
df_limpo = df_limpo[(df_limpo["Year"] >= 1980) & (df_limpo["Year"] <= 2016)].copy()

# agrupando as 31 plataformas em familias para facilitar a leitura das geracoes
def classificar_familia(plat):
    if plat in ["PS", "PS2", "PS3", "PS4", "PSP", "PSV"]:
        return "PlayStation (Sony)"
    elif plat in ["NES", "SNES", "N64", "GC", "Wii", "WiiU"]:
        return "Nintendo (Console de Mesa)"
    elif plat in ["GB", "GBA", "DS", "3DS"]:
        return "Nintendo (Portatil)"
    elif plat in ["XB", "X360", "XOne"]:
        return "Xbox (Microsoft)"
    elif plat == "PC":
        return "PC"
    else:
        return "Sega / Atari / Outros"

df_limpo["Familia"] = df_limpo["Platform"].apply(classificar_familia)

print(f"Registros mantidos (1980-2016): {len(df_limpo)}")
print(f"Vendas globais somadas: {df_limpo['Global_Sales'].sum():.2f} milhoes de copias")

### Resultado da Preparação dos Dados

Após a limpeza, restaram **16.323 títulos** (98,3% da base original), somando **8.820,02 milhões de cópias vendidas** globalmente. A coluna derivada `Familia` agrupa as 31 plataformas individuais em 6 ecossistemas principais, o que permite enxergar a disputa histórica entre Sony, Nintendo, Microsoft, PC e fabricantes clássicas (Sega e Atari) sem poluir a legenda dos gráficos temporais.

In [ ]:
# 1. Evolucao das vendas globais por familia de plataforma (1985-2016)
paleta_familias = {
    "PlayStation (Sony)": "#2563eb",
    "Nintendo (Console de Mesa)": "#dc2626",
    "Nintendo (Portatil)": "#f59e0b",
    "Xbox (Microsoft)": "#16a34a",
    "PC": "#7c3aed",
    "Sega / Atari / Outros": "#64748b"
}

df_tempo = (
    df_limpo[df_limpo["Year"] >= 1985]
    .groupby(["Year", "Familia"], as_index=False)["Global_Sales"]
    .sum()
)

fig1 = px.area(
    df_tempo,
    x="Year",
    y="Global_Sales",
    color="Familia",
    color_discrete_map=paleta_familias,
    labels={
        "Year": "Ano de Lançamento",
        "Global_Sales": "Vendas Globais (milhões de cópias)",
        "Familia": "Família de Plataforma"
    },
    title="1. Evolução das Vendas Globais por Família de Plataforma (1985–2016)"
)
fig1.update_traces(mode="lines", hovertemplate="<b>%{fullData.name}</b><br>Ano: %{x}<br>Vendas: %{y:.2f}M<extra></extra>")
fig1.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=460
)
fig1.add_vline(x=2008, line_dash="dash", line_color="#334155", annotation_text="Pico histórico: 2008 (678.9M)", annotation_position="top left")
fig1.show()

### Análise da Visualização 1 (Descoberta 1: O Pico de Mídia Física entre 2006 e 2011)

O gráfico de área empilhada mostra a transição de liderança tecnológica ao longo de três décadas. Nos anos 1980 e início dos 1990, a Nintendo dominava quase sozinha com o NES, SNES e Game Boy. A partir de 1996, a entrada da Sony (PlayStation 1 e posteriormente PS2) expandiu o tamanho total do mercado.

A principal descoberta dessa série histórica está concentrada na janela de **2006 a 2011**: apenas esses 6 anos responderam por **40,8% de todas as vendas registradas em 37 anos de base** (3.594,81 milhões de cópias de um total de 8.820,02 milhões). O recorde histórico ocorreu em **2008**, com **678,90 milhões de unidades vendidas**, seguido pelo recorde de lançamentos em **2009** (1.431 títulos). Esse salto foi alimentado pela expansão do público casual no Nintendo Wii e Nintendo DS somada à disputa direta entre PlayStation 3 e Xbox 360. Após 2011, observa-se uma retração contínua nas vendas catalogadas, explicada pela migração do consumo para mídia digital (Steam, PS Store, Xbox Live) e dispositivos móveis, que não entram na contagem de varejo físico do VGChartz.

In [ ]:
# 2. Divisao regional de vendas por genero (barras horizontais 100% empilhadas)
genero_reg = df_limpo.groupby("Genre")[["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]].sum()
genero_reg["NA_%"] = genero_reg["NA_Sales"] / genero_reg["Global_Sales"] * 100
genero_reg["EU_%"] = genero_reg["EU_Sales"] / genero_reg["Global_Sales"] * 100
genero_reg["JP_%"] = genero_reg["JP_Sales"] / genero_reg["Global_Sales"] * 100
genero_reg["Outros_%"] = genero_reg["Other_Sales"] / genero_reg["Global_Sales"] * 100
genero_reg = genero_reg.sort_values(by="JP_%", ascending=True)

df_reg_long = genero_reg.reset_index().melt(
    id_vars=["Genre", "Global_Sales"],
    value_vars=["NA_%", "EU_%", "JP_%", "Outros_%"],
    var_name="Regiao",
    value_name="Percentual"
)
mapa_nomes_reg = {
    "NA_%": "América do Norte",
    "EU_%": "Europa",
    "JP_%": "Japão",
    "Outros_%": "Resto do Mundo"
}
df_reg_long["Regiao_Nome"] = df_reg_long["Regiao"].map(mapa_nomes_reg)

fig2 = px.bar(
    df_reg_long,
    x="Percentual",
    y="Genre",
    color="Regiao_Nome",
    orientation="h",
    color_discrete_map={
        "América do Norte": "#1d4ed8",
        "Europa": "#0ea5e9",
        "Japão": "#e11d48",
        "Resto do Mundo": "#94a3b8"
    },
    labels={
        "Percentual": "Participação nas Vendas do Gênero (%)",
        "Genre": "Gênero",
        "Regiao_Nome": "Mercado"
    },
    title="2. Divisão Regional de Vendas por Gênero (Ordenado pelo Peso do Mercado Japonês)"
)
fig2.update_traces(hovertemplate="<b>%{y}</b> — %{fullData.name}<br>Fatia: %{x:.1f}%<extra></extra>")
fig2.update_layout(
    barmode="stack",
    template="plotly_white",
    xaxis=dict(ticksuffix="%", range=[0, 100]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=480
)
fig2.show()

### Análise da Visualização 2 (Descoberta 2: Contraste Cultural entre Japão e Ocidente)

Ao normalizar as vendas de cada gênero em 100% e ordenar pela fatia japonesa, fica evidente uma divisão cultural profunda entre o mercado asiático e o ocidental:

- No gênero **Role-Playing (RPG)**, o Japão sozinho absorve **37,9% de todas as vendas mundiais** (350,2 milhões de 923,8 milhões de cópias), superando tanto a América do Norte (35,3%) quanto a Europa (20,3%). É o único gênero da indústria em que o mercado japonês lidera o consumo global, puxado por franquias locais como *Pokémon*, *Final Fantasy*, *Dragon Quest* e *Monster Hunter*.
- No extremo oposto, em **Shooter** (jogos de tiro), a participação japonesa cai para apenas **3,7%**, enquanto a América do Norte (56,0%) e a Europa (30,3%) concentram **86,3%** de todas as cópias vendidas. Comportamento semelhante ocorre em **Racing** (corrida, com apenas 7,8% no Japão e forte peso europeu de 32,5%) e **Action** (9,2% no Japão contra 80,0% somados entre América do Norte e Europa).

In [ ]:
# 3. Treemap hierarquico interativo: Genero -> Plataforma -> Top 5 Jogos
top_jogos_plat = (
    df_limpo.sort_values("Global_Sales", ascending=False)
    .groupby(["Genre", "Platform"], as_index=False)
    .head(5)
)

fig3 = px.treemap(
    top_jogos_plat,
    path=[px.Constant("Mercado Global (Top 5 por Plataforma/Gênero)"), "Genre", "Platform", "Name"],
    values="Global_Sales",
    color="Genre",
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={"Global_Sales": "Vendas Globais (M)", "Genre": "Gênero"},
    title="3. Mapa Hierárquico de Sucessos: Gênero → Plataforma → Títulos Líderes (Clique para Explorar)"
)
fig3.update_traces(
    hovertemplate="<b>%{label}</b><br>Vendas acumuladas: %{value:.2f} milhões<extra></extra>",
    root_color="#f1f5f9"
)
fig3.update_layout(
    template="plotly_white",
    margin=dict(t=60, l=20, r=20, b=20),
    height=540
)
fig3.show()

### Análise da Visualização 3 (Estrutura Hierárquica por Gênero e Console)

O Treemap interativo permite navegar em três níveis hierárquicos (`Genre -> Platform -> Name`). Ao explorar os blocos, percebe-se que os gêneros têm estruturas de liderança muito distintas:

- Em **Sports**, **Platform** e **Role-Playing**, o topo das vendas é fortemente concentrado em consoles da Nintendo (Wii, DS, NES, Game Boy) e em títulos próprios da fabricante, como *Wii Sports* (82,74M), *Super Mario Bros.* (40,24M) e *Pokémon Red/Blue* (31,37M).
- Já em **Action** e **Shooter**, o volume dos líderes se divide de maneira quase simétrica entre os consoles de mesa da Sony (PS2, PS3, PS4) e da Microsoft (Xbox 360), puxado por franquias multiplataforma como *Grand Theft Auto* (*GTA V* no PS3 com 21,40M e *GTA: San Andreas* no PS2 com 20,81M) e *Call of Duty* (*Modern Warfare 3* e *Black Ops* no X360 e PS3).

In [ ]:
# 4. Boxplot em escala logaritmica + destaque para os megassucessos (>= 15M)
ordem_generos = (
    df_limpo.groupby("Genre")["Global_Sales"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig4 = go.Figure()
for gen in ordem_generos:
    sub = df_limpo[df_limpo["Genre"] == gen]
    fig4.add_trace(
        go.Box(
            y=sub["Global_Sales"],
            x=[gen] * len(sub),
            name=gen,
            boxpoints=False,
            line=dict(width=1.5),
            hoverinfo="y+name"
        )
    )

hits = df_limpo[df_limpo["Global_Sales"] >= 15.0].copy()
fig4.add_trace(
    go.Scatter(
        x=hits["Genre"],
        y=hits["Global_Sales"],
        mode="markers",
        name="Megassucessos (≥ 15M cópias)",
        marker=dict(size=9, color="#dc2626", symbol="diamond", line=dict(width=1, color="white")),
        text=hits["Name"] + " (" + hits["Platform"] + ", " + hits["Year"].astype(str) + ")",
        hovertemplate="<b>%{text}</b><br>Gênero: %{x}<br>Vendas Globais: %{y:.2f}M<extra></extra>"
    )
)

fig4.update_layout(
    title="4. A Cauda Longa dos Jogos: Distribuição por Gênero (Escala Log) e Megassucessos",
    template="plotly_white",
    yaxis=dict(
        title="Vendas Globais em Milhões (Escala Logarítmica)",
        type="log",
        tickvals=[0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 80],
        ticktext=["0.01M", "0.05M", "0.1M", "0.2M", "0.5M", "1M", "2M", "5M", "10M", "20M", "50M", "80M"]
    ),
    xaxis=dict(title="Gênero (ordenado pela mediana de vendas)"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=500
)
fig4.show()

# verificando as metricas de concentracao
mediana = df_limpo["Global_Sales"].median()
media = df_limpo["Global_Sales"].mean()
abaixo_500k = (df_limpo["Global_Sales"] < 0.5).mean() * 100
n_top1 = int(len(df_limpo) * 0.01)
pct_top1 = df_limpo.nlargest(n_top1, "Global_Sales")["Global_Sales"].sum() / df_limpo["Global_Sales"].sum() * 100
n_top5 = int(len(df_limpo) * 0.05)
pct_top5 = df_limpo.nlargest(n_top5, "Global_Sales")["Global_Sales"].sum() / df_limpo["Global_Sales"].sum() * 100

print(f"Mediana global: {mediana:.2f}M | Media global: {media:.2f}M")
print(f"Jogos abaixo de 500 mil copias: {abaixo_500k:.1f}%")
print(f"Top 1% ({n_top1} jogos) = {pct_top1:.1f}% das vendas | Top 5% ({n_top5} jogos) = {pct_top5:.1f}% das vendas")

### Análise da Visualização 4 (Descoberta 3: A Cauda Longa e a Assimetria Extrema)

O Boxplot em escala logarítmica mostra por que avaliar o mercado de jogos apenas pela média aritmética (0,54 milhão de cópias) leva a conclusões equivocadas: **75,7% de todos os jogos catalogados vendem menos de 500 mil cópias**, e a **mediana global é de apenas 0,17 milhão (170 mil unidades)**.

Na prática, a indústria opera sob um regime de concentração severa: o **top 1% dos jogos (apenas 163 títulos)** responde sozinho por **21,1% de todas as cópias vendidas na história da base** (1.861,0 milhões de unidades), e o **top 5% (816 títulos)** concentra **44,6%** de todo o mercado (3.930,8 milhões de unidades). Mesmo dentro de gêneros de alta mediana como *Platform* (0,28M) e *Shooter* (0,23M), a diferença entre um lançamento mediano e um campeão de vendas passa de 100 vezes.

---

## Justificativa das Ferramentas e Limitações dos Dados

**Escolha das ferramentas e representações visuais:** Utilizamos **Python (Pandas + Plotly)** porque permite documentar todo o código de limpeza de maneira auditável e gerar gráficos interativos que funcionam em qualquer navegador (sem exigir instalação de softwares pagos). Os 4 tipos de visualização foram selecionados de acordo com a natureza de cada pergunta: (1) o gráfico de área empilhada para representar evolução contínua no tempo e volume acumulado; (2) as barras horizontais 100% empilhadas para comparar proporções regionais independente do tamanho bruto de cada gênero; (3) o Treemap interativo para navegar pela hierarquia Gênero → Console → Jogo; e (4) o Boxplot em escala logarítmica com pontos sobrepostos para revelar a mediana real sem esconder os grandes *outliers*.

**Limitações do dataset:** (1) os dados do VGChartz cobrem apenas vendas de mídia física em varejo, o que explica a queda aparente após 2011 causada pela migração para compras digitais (Steam, PS Store, Xbox Live); (2) como a base só inclui títulos acima de ~100 mil cópias, milhares de jogos independentes menores ficam de fora; (3) jogos que vinham inclusos na caixa do console (*bundles*, como *Wii Sports* com 82,74M e *Super Mario Bros.* com 40,24M) têm números inflados em relação a títulos vendidos separadamente; e (4) o dataset não contabiliza receitas digitais recorrentes (DLCs, microtransações e assinaturas).

---

### Q&A
- **Como o volume de vendas evoluiu entre as gerações de consoles?** O mercado cresceu gradualmente entre 1985 e 2005 e atingiu uma bolha de vendas físicas entre 2006 e 2011 (40,8% de todo o volume histórico), com pico em 2008 (678,90M de cópias) puxado por Wii, DS, PS3 e Xbox 360.
- **O perfil de consumo japonês difere do ocidental?** Sim. O Japão responde por 37,9% das vendas globais de RPGs (liderando o mundo nesse gênero), mas representa apenas 3,7% das vendas de Shooters, onde América do Norte (56,0%) e Europa (30,3%) dominam.
- **Quais plataformas e franquias sustentam cada gênero?** A Nintendo domina o topo dos jogos individuais em Sports, Platform e RPG com títulos próprios nos consoles Wii, DS, NES e Game Boy, enquanto Action e Shooter têm liderança dividida entre PS2, PS3 e Xbox 360 com franquias como *Grand Theft Auto* e *Call of Duty*.
- **As vendas são bem distribuídas ou concentradas?** Altamente concentradas: 75,7% dos jogos vendem menos de 500 mil cópias (mediana de 0,17M), enquanto o top 5% dos títulos concentra 44,6% de todas as vendas da história da base.

### Data Analysis Key Findings
- Entre 2006 e 2011 foram vendidas 3.594,81 milhões de cópias físicas, o equivalente a 40,8% de todo o volume acumulado nos 37 anos analisados (8.820,02 milhões), com pico de receita física em 2008 (678,90 milhões) e pico de lançamentos em 2009 (1.431 títulos).
- O gênero Role-Playing tem 37,9% de suas vendas globais concentradas no Japão (350,2 milhões de 923,8 milhões), ao passo que o gênero Shooter tem apenas 3,7% de participação japonesa contra 56,0% na América do Norte e 30,3% na Europa.
- A distribuição de vendas por título apresenta forte assimetria positiva: a média é de 0,54 milhão de cópias, mas a mediana é de apenas 0,17 milhão, com o top 1% dos jogos (163 títulos) concentrando 21,1% das vendas globais e o top 5% (816 títulos) concentrando 44,6%.

### Insights or Next Steps
- Para atualizar a análise para o cenário pós-2016, o próximo passo recomendado é cruzar esta base histórica de mídia física com dados de plataformas digitais (como a API pública da Steam/SteamSpy), permitindo medir a substituição do varejo físico pelo digital.
- Do ponto de vista de estratégia de publicação, estúdios focados em RPGs tradicionais encontram no Japão um mercado quase tão grande quanto toda a América do Norte, enquanto projetos de gêneros como Shooter e Racing dependem majoritariamente de localização e marketing voltados para América do Norte e Europa.